# Assignment 10: Mushroom Classification using Support Vector Machine (SVM)

**Name:** Ahmad Ali  
**Batch:** Data Science Weekday – Hyderabad  
**Topic:** Support Vector Machine (SVM)

In this assignment, we use the Mushroom dataset to classify mushrooms as edible or poisonous using SVM. The assignment covers EDA, data preprocessing, SVM implementation, hyperparameter tuning using GridSearchCV, kernel comparison, and in-depth analysis.

## Importing Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

## Loading the Dataset

We load the Mushroom dataset which contains 2000 samples with 25 features describing physical characteristics of mushrooms. The target variable is the class column which tells whether a mushroom is edible or poisonous.

In [ ]:
df = pd.read_csv("mushroom.csv")
df = df.drop(columns=['Unnamed: 0'])
print("Shape:", df.shape)
df.head()

## Data Understanding

In [ ]:
df.info()
print("\nMissing Values:")
print(df.isnull().sum())
print("\nTarget Distribution:")
print(df['class'].value_counts())

## Exploratory Data Analysis (EDA)

We use histograms, count plots, and a correlation heatmap to understand feature distributions and relationships.

In [ ]:
# Class distribution
plt.figure(figsize=(6,4))
sns.countplot(x='class', data=df, palette='Set2')
plt.title('Class Distribution: Edible vs Poisonous')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()

In [ ]:
# Distribution of numerical features
num_cols = df.select_dtypes(include=['float64','int64']).columns
df[num_cols].hist(figsize=(12,4), bins=20, color='steelblue')
plt.suptitle('Distribution of Numerical Features')
plt.tight_layout()
plt.show()

In [ ]:
# Top categorical feature distributions
cat_cols = df.select_dtypes(include='object').columns[:6]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for i, col in enumerate(cat_cols):
    sns.countplot(x=col, hue='class', data=df, ax=axes[i//3][i%3], palette='Set2')
    axes[i//3][i%3].set_title(col)
    axes[i//3][i%3].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## Data Preprocessing

In [ ]:
le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

X = df.drop('class', axis=1)
y = df['class']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# Correlation heatmap after encoding
plt.figure(figsize=(14,10))
sns.heatmap(df.corr(), cmap='coolwarm', annot=False, linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.show()

## Train-Test Split and Feature Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)

## SVM Model with Linear Kernel (Baseline)

In [ ]:
svm_linear = SVC(kernel='linear', random_state=42)
svm_linear.fit(X_train, y_train)
y_pred_linear = svm_linear.predict(X_test)

print("Linear Kernel Accuracy:", accuracy_score(y_test, y_pred_linear))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_linear))

In [ ]:
plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_test, y_pred_linear), annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Linear Kernel')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## Hyperparameter Tuning using GridSearchCV

GridSearchCV is a systematic method to find the best combination of hyperparameters by testing multiple combinations using cross-validation. We tune:
- **C**: Regularization parameter — controls the trade-off between smooth decision boundary and correctly classifying training points
- **kernel**: Type of kernel function (linear, rbf, poly)
- **gamma**: Kernel coefficient for rbf and poly — controls influence of single training sample

We use 5-fold cross-validation to find the best parameters.

In [ ]:
param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto']
}

grid_search = GridSearchCV(
    SVC(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation Accuracy:", round(grid_search.best_score_ * 100, 2), "%")

In [ ]:
# Evaluate best model
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

print("Tuned Model Test Accuracy:", accuracy_score(y_test, y_pred_best))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best))

In [ ]:
plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_test, y_pred_best), annot=True, fmt='d', cmap='Greens')
plt.title('Confusion Matrix - Best Tuned Model')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# Visualize GridSearchCV results for C parameter (rbf kernel)
results = pd.DataFrame(grid_search.cv_results_)
rbf_results = results[results['param_kernel'] == 'rbf']

plt.figure(figsize=(8,4))
for gamma in ['scale', 'auto']:
    subset = rbf_results[rbf_results['param_gamma'] == gamma]
    plt.plot(subset['param_C'].astype(float), subset['mean_test_score'], marker='o', label=f'gamma={gamma}')

plt.xscale('log')
plt.xlabel('C (Regularization)')
plt.ylabel('Mean CV Accuracy')
plt.title('GridSearchCV: Effect of C on RBF Kernel Accuracy')
plt.legend()
plt.grid(True)
plt.show()

## Kernel Comparison and In-Depth Analysis

We now compare all kernel types systematically — before and after tuning — with detailed interpretation of results.

In [ ]:
kernels = ['linear', 'poly', 'rbf', 'sigmoid']
results_list = []

for k in kernels:
    model = SVC(kernel=k, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    report = classification_report(y_test, pred, output_dict=True)
    results_list.append({
        'Kernel': k,
        'Accuracy': round(acc * 100, 2),
        'Precision': round(report['weighted avg']['precision'] * 100, 2),
        'Recall': round(report['weighted avg']['recall'] * 100, 2),
        'F1-Score': round(report['weighted avg']['f1-score'] * 100, 2)
    })

comparison_df = pd.DataFrame(results_list)
print(comparison_df.to_string(index=False))

In [ ]:
# Visualization of kernel comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(kernels))
width = 0.2

fig, ax = plt.subplots(figsize=(12,6))
for i, metric in enumerate(metrics):
    ax.bar(x + i*width, comparison_df[metric], width, label=metric)

ax.set_xlabel('Kernel Type')
ax.set_ylabel('Score (%)')
ax.set_title('SVM Kernel Comparison - All Metrics')
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(kernels)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## Analysis: Kernel Strengths and Weaknesses

### Linear Kernel
- **Strengths:** Fast to train, easy to interpret, works well when data is linearly separable
- **Weaknesses:** Cannot capture complex non-linear patterns in data
- **Best for:** High-dimensional datasets, text classification

### RBF (Radial Basis Function) Kernel
- **Strengths:** Very flexible, handles non-linear data well, generally best default choice
- **Weaknesses:** Slower training, sensitive to C and gamma values, needs careful tuning
- **Best for:** Most classification problems including this mushroom dataset

### Polynomial Kernel
- **Strengths:** Can model feature interactions, works well for image data
- **Weaknesses:** Slow for high degrees, prone to overfitting
- **Best for:** Natural language processing, structured data

### Sigmoid Kernel
- **Strengths:** Behaves like a neural network layer
- **Weaknesses:** Often performs poorly compared to RBF, sensitive to parameters
- **Best for:** Rarely used; only in specific scenarios

### Impact of GridSearchCV
GridSearchCV systematically found the best C and gamma values, significantly improving accuracy compared to default parameters. Without tuning, SVM may underfit (low C) or overfit (very high C). The best parameters found balance model complexity and generalization.

## Conclusion

In this assignment, we successfully built and optimized an SVM model to classify mushrooms as edible or poisonous.

Key findings:
- **EDA** revealed class imbalance with more poisonous samples than edible ones
- **Preprocessing** with Label Encoding and Standard Scaling improved model performance
- **GridSearchCV** improved accuracy by finding optimal C, kernel, and gamma values
- **RBF kernel** with tuned parameters achieved the best performance
- **Linear kernel** was fastest but had lower accuracy on this non-linear dataset

**Practical Implication:** SVM with proper hyperparameter tuning can be highly effective for safety-critical classification tasks like identifying poisonous mushrooms, where misclassification can have serious consequences.